In [1]:
spark

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
0,application_1784712984240_0001,pyspark,idle,Link,Link,✔


SparkSession available as 'spark'.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

import logging

In [3]:
gold_song_charts = spark.read.parquet(
    "s3://group-1-dbda/silver/song_charts/"
)

In [4]:
print("Rows:", gold_song_charts.count())
print("Columns:", len(gold_song_charts.columns))

('Rows:', 42756826)
('Columns:', 32)

In [6]:
gold_song_charts.printSchema()

root
 |-- date: date (nullable = true)
 |-- market: string (nullable = true)
 |-- rank: long (nullable = true)
 |-- uri: string (nullable = true)
 |-- artist_names: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- label: string (nullable = true)
 |-- peak_rank: long (nullable = true)
 |-- previous_rank: long (nullable = true)
 |-- days_on_chart: long (nullable = true)
 |-- streams: long (nullable = true)
 |-- consecutive_days: long (nullable = true)
 |-- entry_status: string (nullable = true)
 |-- peak_date: date (nullable = true)
 |-- entry_rank: long (nullable = true)
 |-- entry_date: date (nullable = true)
 |-- release_date: date (nullable = true)
 |-- artist_uris: string (nullable = true)
 |-- valid_release_date: date (nullable = true)
 |-- month: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- week: integer (nullable = true)
 |-- song_age_days: integer (nullable = true)
 |-- song_age_category: string (nullable = true)
 |-- rank_movemen

In [8]:
gold_song_charts.groupBy("hit_category") \
       .count() \
       .orderBy(desc("count")) \
       .show(truncate=False)

+--------------+--------+
|hit_category  |count   |
+--------------+--------+
|Charting Track|20617881|
|Popular Track |10876724|
|Major Hit     |8992231 |
|Global Hit    |2269990 |
+--------------+--------+

In [9]:
countDistinct("artist_uris")

Column<count(DISTINCT artist_uris)>

In [11]:
gold_song_charts.select("artist_uris").show(10, truncate=False)

+-----------------------------------------------------------------------------------------------------------------+
|artist_uris                                                                                                      |
+-----------------------------------------------------------------------------------------------------------------+
|spotify:artist:4MCBfE4596Uoi2O4DtmEMz                                                                            |
|spotify:artist:72OaDtakiy6yFqkt4TsiFt                                                                            |
|spotify:artist:7rBed6Ya7Hwa2fXbh5btJE                                                                            |
|spotify:artist:1EmdfupUQDpXOcb4Nj2mBH                                                                            |
|spotify:artist:11kt6ggsdxvI8MhyeSMKom                                                                            |
|spotify:artist:6kbR2eL4hecj3rFwGOsYsI|spotify:artist:20rKUmFZsfv9GBXiv6

In [13]:
gold_song_charts.select(
    countDistinct("uri").alias("active_songs")
).show()

+------------+
|active_songs|
+------------+
|      241874|
+------------+

In [14]:
gold_song_charts.agg(
    sum("streams").alias("total_streams")
).show()

+-------------+
|total_streams|
+-------------+
|2898912770514|
+-------------+

In [15]:
gold_song_charts.agg(
    countDistinct("standardized_label").alias("active_labels")
).show()

+-------------+
|active_labels|
+-------------+
|        27381|
+-------------+

In [16]:
gold_song_charts.agg(
    countDistinct("country_name").alias("countries_covered")
).show()

+-----------------+
|countries_covered|
+-----------------+
|               73|
+-----------------+

KPI Cards

In [17]:
from pyspark.sql.functions import split, explode, trim, countDistinct

active_artists = (
    gold_song_charts
    .select(explode(split(col("artist_uris"), "\\|")).alias("artist_uri"))
    .withColumn("artist_uri", trim(col("artist_uri")))
    .filter(col("artist_uri") != "")
    .agg(
        countDistinct("artist_uri").alias("active_artists")
    )
)

active_artists.show()

+--------------+
|active_artists|
+--------------+
|         59661|
+--------------+

In [18]:
gold_song_charts.filter(
    col("hit_category") == "Global Hit"
).agg(
    countDistinct("uri").alias("global_hits")
).show()

+-----------+
|global_hits|
+-----------+
|      32323|
+-----------+

In [19]:
gold_song_charts.filter(
    col("hit_category") == "Major Hit"
).agg(
    countDistinct("uri").alias("major_hits")
).show()

+----------+
|major_hits|
+----------+
|     92299|
+----------+

In [99]:
# KPI Aggregation
from pyspark.sql.functions import *

dashboard_summary = (
    gold_song_charts
    .groupBy("year", "month")
    .agg(
        sum("streams").alias("total_streams"),

        countDistinct("uri").alias("active_songs"),

        countDistinct("standardized_label").alias("active_labels"),

        countDistinct("country_name").alias("countries_covered"),

        countDistinct(
            when(
                col("hit_category").isin("Global Hit", "Major Hit"),
                col("uri")
            )
        ).alias("hit_songs")
    )
)

In [100]:
# Artist Aggregation
artist_summary = (
    gold_song_charts
    .select(
        "year",
        "month",
        explode(split(col("artist_uris"), "\\|")).alias("artist_uri")
    )
    .withColumn(
        "artist_uri",
        trim(col("artist_uri"))
    )
    .filter(col("artist_uri") != "")
    .groupBy("year", "month")
    .agg(
        countDistinct("artist_uri").alias("active_artists")
    )
)

In [101]:
# Join Both Tables
dashboard_summary = (
    dashboard_summary
    .join(
        artist_summary,
        ["year", "month"],
        "left"
    )
)

In [102]:
# Catalog Hit Rate
dashboard_summary = (
    dashboard_summary
    .withColumn(
        "catalog_hit_rate",
        round(
            (
                col("hit_songs")
                / col("active_songs")
            ) * 100,
            2
        )
    )
)

In [103]:
# Order the Data
dashboard_summary = (
    dashboard_summary
    .orderBy("year", "month")
)

Validate

In [25]:
dashboard_summary.show(20, truncate=False)

+----+-----+-------------+------------+-------------+-----------------+-----------+----------+--------------+----------------+
|year|month|total_streams|active_songs|active_labels|countries_covered|global_hits|major_hits|active_artists|catalog_hit_rate|
+----+-----+-------------+------------+-------------+-----------------+-----------+----------+--------------+----------------+
|2017|1    |12662187107  |4298        |1249         |56               |194        |746       |3003          |21.87           |
|2017|2    |12493564382  |4205        |1172         |56               |202        |723       |2914          |22.0            |
|2017|3    |15782903943  |4224        |1158         |56               |204        |779       |2816          |23.27           |
|2017|4    |14600349904  |4025        |1124         |56               |200        |788       |2745          |24.55           |
|2017|5    |15510070927  |4232        |1161         |56               |200        |809       |2878          |23

In [26]:
dashboard_summary.printSchema()

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- total_streams: long (nullable = true)
 |-- active_songs: long (nullable = false)
 |-- active_labels: long (nullable = false)
 |-- countries_covered: long (nullable = false)
 |-- global_hits: long (nullable = false)
 |-- major_hits: long (nullable = false)
 |-- active_artists: long (nullable = true)
 |-- catalog_hit_rate: double (nullable = true)

In [27]:
print("Rows:", dashboard_summary.count())

('Rows:', 113)

In [104]:
dashboard_summary = dashboard_summary.withColumn(
    "year_month",
    date_format(
        to_date(
            concat_ws("-", col("year"), col("month"), lit(1))
        ),
        "MMM yyyy"
    )
)

In [105]:
dashboard_summary = dashboard_summary.select(
    "year",
    "month",
    "year_month",
    "total_streams",
    "active_songs",
    "active_artists",
    "active_labels",
    "countries_covered",
    "hit_songs",
    "catalog_hit_rate"
)

In [106]:
dashboard_summary.printSchema()

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- year_month: string (nullable = true)
 |-- total_streams: long (nullable = true)
 |-- active_songs: long (nullable = false)
 |-- active_artists: long (nullable = true)
 |-- active_labels: long (nullable = false)
 |-- countries_covered: long (nullable = false)
 |-- hit_songs: long (nullable = false)
 |-- catalog_hit_rate: double (nullable = true)

In [107]:
GOLD_DASHBOARD_SUMMARY = "s3://group-1-dbda/gold/dashboard_summary/"

(
    dashboard_summary
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("compression", "snappy")
    .parquet(GOLD_DASHBOARD_SUMMARY)
)

Visual 2

In [108]:
# Aggregate Country Metrics
from pyspark.sql.functions import *

country_summary = (
    gold_song_charts
    .groupBy(
        "year",
        "month",
        "country_name"
    )
    .agg(
        sum("streams").alias("total_streams"),

        countDistinct("uri").alias("active_songs"),

        countDistinct("standardized_label").alias("active_labels")
    )
)

In [32]:
country_summary.show(10, truncate=False)

+----+-----+------------+-------------+------------+-------------+
|year|month|country_name|total_streams|active_songs|active_labels|
+----+-----+------------+-------------+------------+-------------+
|2024|5    |Brazil      |2111148057   |396         |138          |
|2024|11   |Switzerland |86907960     |488         |215          |
|2022|9    |Sweden      |210154864    |362         |155          |
|2022|12   |Netherlands |386625652    |598         |206          |
|2018|12   |Germany     |759633935    |514         |199          |
|2021|12   |Norway      |176633243    |470         |195          |
|2021|6    |Canada      |339027996    |390         |146          |
|2025|2    |Singapore   |87123228     |361         |132          |
|2018|3    |Ireland     |56023161     |436         |157          |
|2020|7    |Hong Kong   |29385309     |339         |123          |
+----+-----+------------+-------------+------------+-------------+
only showing top 10 rows

In [33]:
print(country_summary.count())

7496

In [109]:
# Active Artists
country_artists = (
    gold_song_charts
    .select(
        "year",
        "month",
        "country_name",
        explode(
            split(col("artist_uris"), "\\|")
        ).alias("artist_uri")
    )
    .withColumn(
        "artist_uri",
        trim(col("artist_uri"))
    )
    .filter(col("artist_uri") != "")
    .groupBy(
        "year",
        "month",
        "country_name"
    )
    .agg(
        countDistinct("artist_uri")
        .alias("active_artists")
    )
)

Market Share

In [110]:
monthly_streams = (
    country_summary
    .groupBy(
        "year",
        "month"
    )
    .agg(
        sum("total_streams")
        .alias("monthly_streams")
    )
)

In [111]:
country_summary = (
    country_summary
    .join(
        monthly_streams,
        ["year", "month"]
    )
)

In [112]:
country_summary = (
    country_summary
    .withColumn(
        "market_share",
        round(
            col("total_streams")
            /
            col("monthly_streams")
            * 100,
            2
        )
    )
)

In [113]:
# Join Active Artists
country_summary = (
    country_summary
    .join(
        country_artists,
        [
            "year",
            "month",
            "country_name"
        ],
        "left"
    )
)

Top Artist

In [114]:
# Aggregate Artist Streams
artist_streams = (
    gold_song_charts
    .groupBy(
        "year",
        "month",
        "country_name",
        "artist_names"
    )
    .agg(
        sum("streams").alias("artist_streams")
    )
)

In [40]:
artist_streams.show(10, truncate=False)

+----+-----+------------+----------------------------------------------------------------------------------+--------------+
|year|month|country_name|artist_names                                                                      |artist_streams|
+----+-----+------------+----------------------------------------------------------------------------------+--------------+
|2022|3    |Global      |Justin Bieber|Daniel Caesar|GIVĒON                                                |33822721      |
|2022|12   |Global      |Vishal-Shekhar|Shilpa Rao|Caralisa Monteiro|Vishal Dadlani|Shekhar Ravjiani|Kumaar|1931669       |
|2022|2    |Global      |Imagine Dragons|JID|Arcane|League of Legends                                      |103371145     |
|2022|3    |Global      |Rauw Alejandro                                                                    |34257562      |
|2022|9    |Global      |Ed Sheeran                                                                        |112374140     |
|2022|12

In [115]:
# Window Specification
from pyspark.sql.window import Window

artist_window = Window.partitionBy(
    "year",
    "month",
    "country_name"
).orderBy(
    desc("artist_streams"),
    asc("artist_names")   # tie-breaker
)


In [116]:
# Rank
top_artist = (
    artist_streams
    .withColumn(
        "rn",
        row_number().over(artist_window)
    )
    .filter(col("rn") == 1)
    .drop("rn")
)

In [117]:
# Rename
top_artist = (
    top_artist
    .withColumnRenamed(
        "artist_names",
        "top_artist"
    )
)

Validate

In [44]:
top_artist.show(20, truncate=False)

+----+-----+------------------+----------------------+--------------+
|year|month|country_name      |top_artist            |artist_streams|
+----+-----+------------------+----------------------+--------------+
|2017|5    |Canada            |Kendrick Lamar        |15256596      |
|2017|11   |Hungary           |Ed Sheeran            |388225        |
|2018|2    |Taiwan            |A-Mei Chang           |1554533       |
|2018|3    |Ireland           |Drake                 |1845552       |
|2018|3    |Philippines       |Moira Dela Torre      |17272386      |
|2018|4    |Colombia          |Ozuna                 |2140886       |
|2018|5    |Belgium           |Post Malone           |1892544       |
|2018|10   |Greece            |Eminem                |377942        |
|2018|12   |Germany           |Capital Bra           |27708610      |
|2019|2    |Estonia           |Ariana Grande         |478012        |
|2019|2    |Philippines       |Ariana Grande         |21781942      |
|2019|5    |Global  

In [45]:
print(top_artist.count())

7496

Top Label

In [118]:
# Aggregate Label Streams
label_streams = (
    gold_song_charts
    .groupBy(
        "year",
        "month",
        "country_name",
        "standardized_label"
    )
    .agg(
        sum("streams").alias("label_streams")
    )
)

In [119]:
# Window
label_window = Window.partitionBy(
    "year",
    "month",
    "country_name"
).orderBy(
    desc("label_streams"),
    asc("standardized_label")
)

In [120]:
# Rank
top_label = (
    label_streams
    .withColumn(
        "rn",
        row_number().over(label_window)
    )
    .filter(col("rn") == 1)
    .drop("rn")
)

In [121]:
# Rename
top_label = (
    top_label
    .withColumnRenamed(
        "standardized_label",
        "top_label"
    )
)

In [50]:
# Validate
top_label.show(20, truncate=False)

+----+-----+------------------+------------------------------+-------------+
|year|month|country_name      |top_label                     |label_streams|
+----+-----+------------------+------------------------------+-------------+
|2017|5    |Canada            |Aftermath                     |20164232     |
|2017|11   |Hungary           |Atlantic Records Uk           |759821       |
|2018|2    |Taiwan            |Atlantic Records Uk           |1925131      |
|2018|3    |Ireland           |Atlantic Records Uk           |4089066      |
|2018|3    |Philippines       |Abs-cbn Film Productions, Inc.|24742478     |
|2018|4    |Colombia          |Umle - Latino                 |11568840     |
|2018|5    |Belgium           |Republic Records              |4323150      |
|2018|10   |Greece            |Eminem Catalog Ps             |579075       |
|2018|12   |Germany           |Urban                         |49075896     |
|2019|2    |Estonia           |Republic Records              |620458       |

In [51]:
print(top_label.count())

7496

In [122]:
# merge everything
country_performance = (
    country_summary
    .join(
        top_artist.select(
            "year",
            "month",
            "country_name",
            "top_artist"
        ),
        ["year", "month", "country_name"],
        "left"
    )
    .join(
        top_label.select(
            "year",
            "month",
            "country_name",
            "top_label"
        ),
        ["year", "month", "country_name"],
        "left"
    )
)

In [58]:
country_performance.printSchema()

print(country_performance.count())

country_performance.show(10, truncate=False)

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- country_name: string (nullable = true)
 |-- total_streams: long (nullable = true)
 |-- active_songs: long (nullable = false)
 |-- active_labels: long (nullable = false)
 |-- monthly_streams: long (nullable = true)
 |-- market_share: double (nullable = true)
 |-- active_artists: long (nullable = true)
 |-- top_artist: string (nullable = true)
 |-- top_label: string (nullable = true)

7496
+----+-----+------------+-------------+------------+-------------+---------------+------------+--------------+----------------+------------------------------+
|year|month|country_name|total_streams|active_songs|active_labels|monthly_streams|market_share|active_artists|top_artist      |top_label                     |
+----+-----+------------+-------------+------------+-------------+---------------+------------+--------------+----------------+------------------------------+
|2017|5    |Canada      |276659588    |326     

In [123]:
country_performance = country_performance.drop("monthly_streams")

In [124]:
country_performance.printSchema()

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- country_name: string (nullable = true)
 |-- total_streams: long (nullable = true)
 |-- active_songs: long (nullable = false)
 |-- active_labels: long (nullable = false)
 |-- market_share: double (nullable = true)
 |-- active_artists: long (nullable = true)
 |-- top_artist: string (nullable = true)
 |-- top_label: string (nullable = true)

In [125]:
# Growth %
# Create the Window
from pyspark.sql.window import Window
from pyspark.sql.functions import lag

growth_window = (
    Window
    .partitionBy("country_name")
    .orderBy("year", "month")
)

In [126]:
# Get Previous Month Streams
country_performance = (
    country_performance
    .withColumn(
        "previous_month_streams",
        lag("total_streams").over(growth_window)
    )
)

In [127]:
# Calculate Growth %
country_performance = (
    country_performance
    .withColumn(
        "growth_percentage",
        round(
            (
                (col("total_streams") - col("previous_month_streams"))
                / col("previous_month_streams")
            ) * 100,
            2
        )
    )
)

In [128]:
# Handle First Month : he first record for each country has no previous month, so growth_percentage will be NULL.
country_performance = (
    country_performance
    .fillna(
        {"growth_percentage": 0.0}
    )
)

In [129]:
# Clean Up
country_performance = (
    country_performance
    .drop("previous_month_streams")
)

In [68]:
# Validate
country_performance.select(
    "year",
    "month",
    "country_name",
    "total_streams",
    "growth_percentage"
).show(20, truncate=False)

+----+-----+------------+-------------+-----------------+
|year|month|country_name|total_streams|growth_percentage|
+----+-----+------------+-------------+-----------------+
|2017|1    |Paraguay    |10371557     |0.0              |
|2017|2    |Paraguay    |10410608     |0.38             |
|2017|3    |Paraguay    |11653846     |11.94            |
|2017|4    |Paraguay    |10599602     |-9.05            |
|2017|5    |Paraguay    |12382668     |16.82            |
|2017|6    |Paraguay    |12201481     |-1.46            |
|2017|7    |Paraguay    |15152134     |24.18            |
|2017|8    |Paraguay    |15083614     |-0.45            |
|2017|9    |Paraguay    |14089464     |-6.59            |
|2017|10   |Paraguay    |13467925     |-4.41            |
|2017|11   |Paraguay    |17077323     |26.8             |
|2017|12   |Paraguay    |19866357     |16.33            |
|2018|1    |Paraguay    |20898634     |5.2              |
|2018|2    |Paraguay    |20510419     |-1.86            |
|2018|3    |Pa

In [130]:
# Calculate Country Hit Counts
country_hits = (
    gold_song_charts
    .groupBy(
        "year",
        "month",
        "country_name"
    )
    .agg(
        countDistinct(
            when(
                col("hit_category").isin("Global Hit", "Major Hit"),
                col("uri")
            )
        ).alias("hit_songs")
    )
)

In [70]:
country_hits.show(10, truncate=False)

+----+-----+-------------+-----------+----------+
|year|month|country_name |global_hits|major_hits|
+----+-----+-------------+-----------+----------+
|2024|11   |Switzerland  |38         |142       |
|2022|9    |Sweden       |20         |89        |
|2022|12   |Netherlands  |29         |148       |
|2018|12   |Germany      |32         |132       |
|2021|12   |Norway       |29         |107       |
|2025|2    |Singapore    |14         |91        |
|2024|4    |Slovakia     |36         |111       |
|2022|12   |Thailand     |21         |72        |
|2018|4    |Colombia     |16         |58        |
|2024|8    |United States|19         |89        |
+----+-----+-------------+-----------+----------+
only showing top 10 rows

In [136]:
print(country_performance.columns)

['year', 'month', 'country_name', 'total_streams', 'active_songs', 'active_labels', 'market_share', 'active_artists', 'top_artist', 'top_label', 'growth_percentage']

In [135]:
country_performance = country_performance.drop("hit_songs")

In [137]:
# Join + Catalog Hit Rate
country_performance = (
    country_performance
    .join(
        country_hits,
        ["year", "month", "country_name"],
        "left"
    )
    .fillna({"hit_songs": 0})
    .withColumn(
        "catalog_hit_rate",
        round(
            col("hit_songs")
            /
            col("active_songs")
            * 100,
            2
        )
    )
    .drop("hit_songs")
)

In [138]:
country_performance.select(
    "country_name",
    "active_songs",
    "catalog_hit_rate"
).show(10, truncate=False)

+------------+------------+----------------+
|country_name|active_songs|catalog_hit_rate|
+------------+------------+----------------+
|Canada      |326         |21.47           |
|Hungary     |426         |19.95           |
|Taiwan      |491         |18.94           |
|Ireland     |436         |17.89           |
|Philippines |290         |22.76           |
|Colombia    |302         |21.52           |
|Belgium     |417         |21.34           |
|Greece      |283         |36.4            |
|Germany     |514         |26.26           |
|Estonia     |124         |74.19           |
+------------+------------+----------------+
only showing top 10 rows

In [139]:
country_performance = country_performance.select(
    "year",
    "month",
    "country_name",
    "total_streams",
    "market_share",
    "growth_percentage",
    "catalog_hit_rate",
    "active_songs",
    "active_artists",
    "active_labels",
    "top_artist",
    "top_label"
)

In [75]:
# Final Validation
country_performance.printSchema()

print("Rows:", country_performance.count())

country_performance.show(20, truncate=False)

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- country_name: string (nullable = true)
 |-- total_streams: long (nullable = true)
 |-- market_share: double (nullable = true)
 |-- growth_percentage: double (nullable = false)
 |-- catalog_hit_rate: double (nullable = true)
 |-- active_songs: long (nullable = false)
 |-- active_artists: long (nullable = true)
 |-- active_labels: long (nullable = false)
 |-- top_artist: string (nullable = true)
 |-- top_label: string (nullable = true)

('Rows:', 7496)
+----+-----+------------------+-------------+------------+-----------------+----------------+------------+--------------+-------------+----------------------+------------------------------+
|year|month|country_name      |total_streams|market_share|growth_percentage|catalog_hit_rate|active_songs|active_artists|active_labels|top_artist            |top_label                     |
+----+-----+------------------+-------------+------------+-----------------+----

In [140]:
country_performance = (
    country_performance
    .withColumn(
        "year_month",
        date_format(
            to_date(
                concat_ws("-", col("year"), col("month"), lit(1))
            ),
            "MMM yyyy"
        )
    )
)

In [141]:
country_performance = country_performance.select(
    "year",
    "month",
    "year_month",
    "country_name",
    "total_streams",
    "market_share",
    "growth_percentage",
    "catalog_hit_rate",
    "active_songs",
    "active_artists",
    "active_labels",
    "top_artist",
    "top_label"
)

In [142]:
GOLD_COUNTRY_PERFORMANCE = "s3://group-1-dbda/gold/country_performance/"

(
    country_performance
    .write
    .mode("overwrite")
    .option("compression", "snappy")
    .partitionBy("year")
    .parquet(GOLD_COUNTRY_PERFORMANCE)
)

Monthly Trend

In [79]:
# Monthly Aggregation
from pyspark.sql.functions import *

monthly_trends = (
    gold_song_charts
    .groupBy(
        "year",
        "month"
    )
    .agg(
        sum("streams").alias("total_streams"),

        countDistinct("uri").alias("active_songs"),

        countDistinct("standardized_label").alias("active_labels")
    )
)

In [80]:
# Active Artists
monthly_artists = (
    gold_song_charts
    .select(
        "year",
        "month",
        explode(split(col("artist_uris"), "\\|")).alias("artist_uri")
    )
    .withColumn(
        "artist_uri",
        trim(col("artist_uri"))
    )
    .filter(col("artist_uri") != "")
    .groupBy(
        "year",
        "month"
    )
    .agg(
        countDistinct("artist_uri").alias("active_artists")
    )
)

In [81]:
# Join
monthly_trends = (
    monthly_trends
    .join(
        monthly_artists,
        ["year", "month"],
        "left"
    )
)

In [82]:
monthly_trends = (
    monthly_trends
    .withColumn(
        "year_month",
        date_format(
            to_date(
                concat_ws("-", col("year"), col("month"), lit(1))
            ),
            "MMM yyyy"
        )
    )
)

In [83]:
monthly_trends = (
    monthly_trends.select(
        "year",
        "month",
        "year_month",
        "total_streams",
        "active_songs",
        "active_artists",
        "active_labels"
    )
)

In [84]:
# Validate
monthly_trends.printSchema()

print("Rows:", monthly_trends.count())

monthly_trends.orderBy("year", "month").show(20, truncate=False)

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- year_month: string (nullable = true)
 |-- total_streams: long (nullable = true)
 |-- active_songs: long (nullable = false)
 |-- active_artists: long (nullable = true)
 |-- active_labels: long (nullable = false)

('Rows:', 113)
+----+-----+----------+-------------+------------+--------------+-------------+
|year|month|year_month|total_streams|active_songs|active_artists|active_labels|
+----+-----+----------+-------------+------------+--------------+-------------+
|2017|1    |Jan 2017  |12662187107  |4298        |3003          |1249         |
|2017|2    |Feb 2017  |12493564382  |4205        |2914          |1172         |
|2017|3    |Mar 2017  |15782903943  |4224        |2816          |1158         |
|2017|4    |Apr 2017  |14600349904  |4025        |2745          |1124         |
|2017|5    |May 2017  |15510070927  |4232        |2878          |1161         |
|2017|6    |Jun 2017  |14770389443  |4471       

In [85]:
GOLD_MONTHLY_TRENDS = "s3://group-1-dbda/gold/monthly_trends/"

(
    monthly_trends
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("compression", "snappy")
    .parquet(GOLD_MONTHLY_TRENDS)
)

Label Treemap

In [143]:
# Aggregate Label Metrics
from pyspark.sql.functions import *

label_summary = (
    gold_song_charts
    .groupBy(
        "year",
        "month",
        "standardized_label"
    )
    .agg(
        sum("streams").alias("total_streams"),

        countDistinct("uri").alias("active_songs")
    )
)

In [144]:
# Active Artists
label_artists = (
    gold_song_charts
    .select(
        "year",
        "month",
        "standardized_label",
        explode(split(col("artist_uris"), "\\|")).alias("artist_uri")
    )
    .withColumn("artist_uri", trim(col("artist_uri")))
    .filter(col("artist_uri") != "")
    .groupBy(
        "year",
        "month",
        "standardized_label"
    )
    .agg(
        countDistinct("artist_uri").alias("active_artists")
    )
)

In [145]:
# Market Share
monthly_label_streams = (
    label_summary
    .groupBy("year", "month")
    .agg(
        sum("total_streams").alias("monthly_streams")
    )
)

label_summary = (
    label_summary
    .join(monthly_label_streams, ["year", "month"])
    .withColumn(
        "market_share",
        round(
            col("total_streams") /
            col("monthly_streams") * 100,
            2
        )
    )
    .drop("monthly_streams")
)

In [146]:
# Catalog Hit Rate
label_hits = (
    gold_song_charts
    .groupBy(
        "year",
        "month",
        "standardized_label"
    )
    .agg(
        countDistinct(
            when(
                col("hit_category").isin("Global Hit", "Major Hit"),
                col("uri")
            )
        ).alias("hit_songs")
    )
)

In [147]:
# Final Join
label_performance = (
    label_summary
    .join(
        label_artists,
        ["year", "month", "standardized_label"],
        "left"
    )
    .join(
        label_hits,
        ["year", "month", "standardized_label"],
        "left"
    )
    .fillna({"hit_songs": 0})
    .withColumn(
        "catalog_hit_rate",
        round(
            col("hit_songs")
            / col("active_songs")
            * 100,
            2
        )
    )
    .drop("hit_songs")
)

In [148]:
label_performance.select(
    "standardized_label",
    "active_songs",
    "catalog_hit_rate"
).orderBy(desc("catalog_hit_rate")).show(20, truncate=False)

+-----------------------------+------------+----------------+
|standardized_label           |active_songs|catalog_hit_rate|
+-----------------------------+------------+----------------+
|Payner                       |10          |100.0           |
|M Music Live                 |1           |100.0           |
|Beluga Heights/warner Records|1           |100.0           |
|Lost Army                    |1           |100.0           |
|Young Boss Entertainment     |1           |100.0           |
|Thaitanium Entertainment     |1           |100.0           |
|Bloodpop/dj/rmbg/republic    |1           |100.0           |
|On Records Team              |1           |100.0           |
|Sony Music/sls Music         |1           |100.0           |
|Petroleum Records            |1           |100.0           |
|Bermudu Divsturis            |1           |100.0           |
|Netīrās Cilpas               |1           |100.0           |
|Cloud 9 Music                |1           |100.0           |
|Left La

In [150]:
label_performance = (
    label_performance
    .withColumn(
        "year_month",
        date_format(
            to_date(
                concat_ws("-", col("year"), col("month"), lit(1))
            ),
            "MMM yyyy"
        )
    )
)

In [151]:
# Final Column Order
label_performance = label_performance.select(
    "year",
    "month",
    "year_month",
    "standardized_label",
    "total_streams",
    "market_share",
    "catalog_hit_rate",
    "active_songs",
    "active_artists"
)

In [152]:
# Validate
label_performance.printSchema()


root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- year_month: string (nullable = true)
 |-- standardized_label: string (nullable = true)
 |-- total_streams: long (nullable = true)
 |-- market_share: double (nullable = true)
 |-- catalog_hit_rate: double (nullable = true)
 |-- active_songs: long (nullable = false)
 |-- active_artists: long (nullable = true)

In [153]:
GOLD_LABEL_PERFORMANCE = "s3://group-1-dbda/gold/label_performance/"

(
    label_performance
    .write
    .mode("overwrite")
    .option("compression", "snappy")
    .partitionBy("year")
    .parquet(GOLD_LABEL_PERFORMANCE)
)

Aggregate Artist Metrics

In [154]:
gold_song_charts.select(
    "artist_uris",
    "artist_names"
).show(10, truncate=False)

+-----------------------------------------------------------------------------------------------------------------+-------------------------------------+
|artist_uris                                                                                                      |artist_names                         |
+-----------------------------------------------------------------------------------------------------------------+-------------------------------------+
|spotify:artist:4MCBfE4596Uoi2O4DtmEMz                                                                            |Juice WRLD                           |
|spotify:artist:72OaDtakiy6yFqkt4TsiFt                                                                            |Cher                                 |
|spotify:artist:7rBed6Ya7Hwa2fXbh5btJE                                                                            |LACAZETTE                            |
|spotify:artist:1EmdfupUQDpXOcb4Nj2mBH                                      

In [156]:
artist_data = (
    gold_song_charts
    .select(
        arrays_zip(
            split(col("artist_uris"), "\\|"),
            split(col("artist_names"), "\\|")
        ).alias("artists")
    )
)

artist_data.printSchema()

root
 |-- artists: array (nullable = true)
 |    |-- element: struct (containsNull = false)
 |    |    |-- 0: string (nullable = true)
 |    |    |-- 1: string (nullable = true)

In [157]:
from pyspark.sql.functions import *

artist_data = (
    gold_song_charts
    .select(
        "year",
        "month",
        "country_name",
        "uri",
        "streams",
        "chart_strength_score",
        "hit_category",
        arrays_zip(
            split(col("artist_uris"), "\\|"),
            split(col("artist_names"), "\\|")
        ).alias("artists")
    )
    .withColumn(
        "artist",
        explode(col("artists"))
    )
    .withColumn("artist_uri", trim(col("artist.0")))
    .withColumn("artist_name", trim(col("artist.1")))
    .drop("artists", "artist")
)

In [158]:
artist_data.printSchema()

artist_data.show(10, truncate=False)

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- country_name: string (nullable = true)
 |-- uri: string (nullable = true)
 |-- streams: long (nullable = true)
 |-- chart_strength_score: double (nullable = true)
 |-- hit_category: string (nullable = true)
 |-- artist_uri: string (nullable = true)
 |-- artist_name: string (nullable = true)

+----+-----+------------+------------------------------------+-------+--------------------+--------------+-------------------------------------+-------------+
|year|month|country_name|uri                                 |streams|chart_strength_score|hit_category  |artist_uri                           |artist_name  |
+----+-----+------------+------------------------------------+-------+--------------------+--------------+-------------------------------------+-------------+
|2025|1    |Germany     |spotify:track:285pBltuF7vW8TeWk8hdRR|69591  |50.88               |Charting Track|spotify:artist:4MCBfE4596Uoi2O4DtmEMz|J

In [159]:
# Aggregate Artist Metrics
artist_summary = (
    artist_data
    .groupBy(
        "year",
        "month",
        "artist_uri",
        "artist_name"
    )
    .agg(
        sum("streams").alias("total_streams"),

        countDistinct("uri").alias("active_songs"),

        countDistinct("country_name").alias("countries_reached"),

        round(
            avg("chart_strength_score"),
            2
        ).alias("avg_chart_strength")
    )
)

In [160]:
# Hit Songs
artist_hits = (
    artist_data
    .groupBy(
        "year",
        "month",
        "artist_uri",
        "artist_name"
    )
    .agg(
        countDistinct(
            when(
                col("hit_category").isin("Global Hit", "Major Hit"),
                col("uri")
            )
        ).alias("hit_songs")
    )
)

In [161]:
# Final Join
artist_performance = (
    artist_summary
    .join(
        artist_hits,
        [
            "year",
            "month",
            "artist_uri",
            "artist_name"
        ],
        "left"
    )
    .fillna({"hit_songs": 0})
    .withColumn(
        "catalog_hit_rate",
        round(
            col("hit_songs")
            / col("active_songs")
            * 100,
            2
        )
    )
    .drop("hit_songs")
)

In [162]:
artist_performance = (
    artist_performance
    .withColumn(
        "year_month",
        date_format(
            to_date(
                concat_ws("-", col("year"), col("month"), lit(1))
            ),
            "MMM yyyy"
        )
    )
)

In [163]:
artist_performance = artist_performance.select(
    "year",
    "month",
    "year_month",
    "artist_uri",
    "artist_name",
    "total_streams",
    "active_songs",
    "countries_reached",
    "catalog_hit_rate",
    "avg_chart_strength"
)

In [164]:
artist_performance.printSchema()

artist_performance.show(10, truncate=False)

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- year_month: string (nullable = true)
 |-- artist_uri: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- total_streams: long (nullable = true)
 |-- active_songs: long (nullable = false)
 |-- countries_reached: long (nullable = false)
 |-- catalog_hit_rate: double (nullable = true)
 |-- avg_chart_strength: double (nullable = true)

+----+-----+----------+-------------------------------------+-----------------+-------------+------------+-----------------+----------------+------------------+
|year|month|year_month|artist_uri                           |artist_name      |total_streams|active_songs|countries_reached|catalog_hit_rate|avg_chart_strength|
+----+-----+----------+-------------------------------------+-----------------+-------------+------------+-----------------+----------------+------------------+
|2017|1    |Jan 2017  |spotify:artist:0FDJB5xf8i09jDjIg1qNED|Carlitos Rossy  

In [165]:
artist_performance.select(
    max("catalog_hit_rate").alias("max_hit_rate")
).show()

+------------+
|max_hit_rate|
+------------+
|       100.0|
+------------+

In [166]:
artist_performance.select(
    max("countries_reached")
).show()

+----------------------+
|max(countries_reached)|
+----------------------+
|                    73|
+----------------------+

In [167]:
GOLD_ARTIST_PERFORMANCE = "s3://group-1-dbda/gold/artist_performance/"

(
    artist_performance
    .write
    .mode("overwrite")
    .option("compression", "snappy")
    .partitionBy("year")
    .parquet(GOLD_ARTIST_PERFORMANCE)
)